# EEG Portfolio 01 — Exploring Annotated Seizure and Non-Seizure Segments

**Author:** Niknaz Akbarpour  
**Program:** Honours Psychology, Neuroscience & Behaviour (B.Sc.), McMaster University

## Research question
How do the time-domain appearance and frequency content of an annotated seizure segment differ from a non-seizure segment within one scalp-EEG recording?

> **Scope statement:** This notebook is an educational, retrospective exploration of a public, de-identified dataset. It is not a medical device and is not intended to diagnose, predict, monitor, or guide treatment of seizures.

## 1. Install and import packages
Run this notebook in Google Colab. The first cell installs MNE, which is used for EEG/MEG analysis.

In [ ]:
%pip -q install mne edfio

from pathlib import Path
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
import mne

mne.set_log_level("WARNING")
print("MNE version:", mne.__version__)

## 2. Download one public EEG recording and its annotations
This notebook uses `chb01_03.edf` from the CHB–MIT Scalp EEG Database. Only one recording is downloaded for this introductory analysis.

In [ ]:
DATA_DIR = Path("data/chb01")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://physionet.org/files/chbmit/1.0.0/chb01"
EDF_NAME = "chb01_03.edf"
SUMMARY_NAME = "chb01-summary.txt"

def download_file(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already available: {destination}")
        return
    print(f"Downloading {destination.name} ...")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(destination, "wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    print(f"Saved: {destination} ({destination.stat().st_size / 1e6:.1f} MB)")

edf_path = DATA_DIR / EDF_NAME
summary_path = DATA_DIR / SUMMARY_NAME
download_file(f"{BASE_URL}/{EDF_NAME}", edf_path)
download_file(f"{BASE_URL}/{SUMMARY_NAME}", summary_path)

## 3. Parse the seizure annotation
The seizure start and end times are read from the dataset's summary file rather than typed manually.

In [ ]:
summary_text = summary_path.read_text(errors="replace")
file_block_match = re.search(
    rf"File Name:\s*{re.escape(EDF_NAME)}(.*?)(?=File Name:|\Z)",
    summary_text,
    flags=re.DOTALL,
)
if file_block_match is None:
    raise ValueError(f"Could not find annotations for {EDF_NAME}.")

file_block = file_block_match.group(1)
start_match = re.search(r"Seizure(?: \d+)? Start Time:\s*(\d+) seconds", file_block)
end_match = re.search(r"Seizure(?: \d+)? End Time:\s*(\d+) seconds", file_block)
if start_match is None or end_match is None:
    raise ValueError("A seizure interval could not be parsed from the summary file.")

seizure_start = int(start_match.group(1))
seizure_end = int(end_match.group(1))
print(f"Annotated seizure: {seizure_start}–{seizure_end} seconds")
print(f"Duration: {seizure_end - seizure_start} seconds")

## 4. Load and describe the EEG recording

In [ ]:
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
duration_seconds = raw.n_times / raw.info["sfreq"]

recording_summary = pd.DataFrame({
    "Characteristic": [
        "File", "Duration (minutes)", "Sampling frequency (Hz)",
        "Number of channels", "Annotated seizure start (s)",
        "Annotated seizure end (s)"
    ],
    "Value": [
        EDF_NAME, round(duration_seconds / 60, 2), raw.info["sfreq"],
        len(raw.ch_names), seizure_start, seizure_end
    ]
})
display(recording_summary)
print("First 10 channel names:", raw.ch_names[:10])

## 5. Prepare comparable 20-second segments
A non-seizure segment near the beginning of the recording is compared with a segment that begins five seconds before the annotated seizure onset. A 0.5–40 Hz band-pass filter is applied for visualization only.

In [ ]:
NON_SEIZURE_START = 60.0
WINDOW_SECONDS = 20.0
PRE_ONSET_SECONDS = 5.0

non_seizure = raw.copy().crop(
    tmin=NON_SEIZURE_START,
    tmax=NON_SEIZURE_START + WINDOW_SECONDS,
).load_data()

seizure_window_start = max(0.0, seizure_start - PRE_ONSET_SECONDS)
seizure_segment = raw.copy().crop(
    tmin=seizure_window_start,
    tmax=seizure_window_start + WINDOW_SECONDS,
).load_data()

for segment in (non_seizure, seizure_segment):
    segment.filter(l_freq=0.5, h_freq=40.0, verbose=False)

selected_channels = [name for name in raw.ch_names if "-" in name][:8]
if len(selected_channels) < 4:
    selected_channels = raw.ch_names[:8]
print("Channels shown:", selected_channels)

In [ ]:
def plot_eeg_segment(segment, channels, title, seizure_interval=None):
    data_uv = segment.get_data(picks=channels) * 1e6
    times = np.arange(data_uv.shape[1]) / segment.info["sfreq"]
    robust_scale = np.nanpercentile(np.abs(data_uv), 95)
    offset = max(robust_scale * 2.5, 50)

    fig, ax = plt.subplots(figsize=(14, 7))
    for index, channel in enumerate(channels):
        ax.plot(times, data_uv[index] + index * offset, linewidth=0.7)
    if seizure_interval is not None:
        ax.axvspan(*seizure_interval, color="#9c27b0", alpha=0.18, label="Annotated seizure")
        ax.legend(loc="upper right")
    ax.set_yticks(np.arange(len(channels)) * offset)
    ax.set_yticklabels(channels)
    ax.set_xlabel("Time within displayed segment (seconds)")
    ax.set_ylabel("EEG channels (vertically offset; µV)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_eeg_segment(
    non_seizure, selected_channels,
    "Example non-seizure EEG segment (filtered 0.5–40 Hz)"
)

relative_seizure_start = seizure_start - seizure_window_start
relative_seizure_end = min(seizure_end - seizure_window_start, WINDOW_SECONDS)
plot_eeg_segment(
    seizure_segment, selected_channels,
    "EEG segment around annotated seizure onset (filtered 0.5–40 Hz)",
    seizure_interval=(relative_seizure_start, relative_seizure_end),
)

## 6. Compare average power spectra
Welch's method estimates the power spectral density (PSD). The plot is descriptive: it does not establish a diagnostic biomarker.

In [ ]:
def mean_psd(segment, channels):
    data = segment.get_data(picks=channels)
    sfreq = segment.info["sfreq"]
    frequencies, power = welch(
        data, fs=sfreq, nperseg=int(2 * sfreq), axis=-1
    )
    keep = (frequencies >= 0.5) & (frequencies <= 40)
    return frequencies[keep], power[:, keep].mean(axis=0)

freq_non, psd_non = mean_psd(non_seizure, selected_channels)
freq_sz, psd_sz = mean_psd(seizure_segment, selected_channels)

plt.figure(figsize=(10, 5))
plt.semilogy(freq_non, psd_non, label="Non-seizure segment", linewidth=2)
plt.semilogy(freq_sz, psd_sz, label="Segment around seizure onset", linewidth=2)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Mean PSD (V²/Hz, log scale)")
plt.title("Average power spectral density across selected channels")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Interpretation prompts
Write two or three sentences for each question after inspecting your output:

1. What visible changes appear around the annotated seizure onset?
2. Are those changes present across every displayed channel or concentrated in particular channels?
3. How does the PSD differ between the two selected segments?
4. Why can these two hand-selected segments not establish generalizable seizure detection?
5. What additional patients, recordings, controls, and validation steps would be needed?

### Suggested cautious conclusion
In this single-recording exploratory comparison, the annotated seizure segment showed observable time- and frequency-domain differences from one selected non-seizure segment. These observations are descriptive and cannot establish diagnostic or predictive performance. Subsequent analysis should use systematically generated windows, patient- or recording-grouped validation, imbalance-aware metrics, and multiple participants.

## Data citation
Guttag, J. (2010). *CHB-MIT Scalp EEG Database* (version 1.0.0). PhysioNet. https://doi.org/10.13026/C2K01R

## Completion checklist
- [ ] All cells run without errors.
- [ ] Recording summary is displayed.
- [ ] Non-seizure EEG figure is generated.
- [ ] Annotated seizure-onset figure is generated.
- [ ] PSD comparison is generated.
- [ ] Interpretation prompts are answered in your own words.
- [ ] No diagnostic, predictive, or clinical-performance claim is made.